# Task 1: Exploratory Data Analysis

This notebook will cover descriptive statistics, keyword and topic analysis, publication frequency, and publisher analysis for the financial news dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data_loader import load_news_data
from src.eda_utils import (
    add_headline_features,
    common_phrases,
    extract_domain,
    publication_counts_by_day,
    publication_counts_by_hour,
    top_keywords,
)

sns.set_theme(style="whitegrid")

DATA_DIR = Path("../data/raw")
NEWS_DATA_PATH = DATA_DIR / "financial_news.csv"

print(f"News data path: {NEWS_DATA_PATH}")
print(f"File exists: {NEWS_DATA_PATH.exists()}")

## Planned analysis

1. Load and clean the financial news dataset.
2. Explore headline length, article counts, and publication trends.
3. Extract keywords and recurring themes.
4. Compare publisher activity and posting patterns.

## 1. Load and Explore Data

In [ ]:
if NEWS_DATA_PATH.exists():
    df_news = load_news_data(str(NEWS_DATA_PATH))
    print("\nDataset shape:", df_news.shape)
    print("\nColumns:", df_news.columns.tolist())
    print("\nFirst few rows:")
    print(df_news.head())
    print("\nData types:")
    print(df_news.dtypes)
    print("\nMissing values:")
    print(df_news.isnull().sum())
else:
    raise FileNotFoundError(
        f"Could not find {NEWS_DATA_PATH}. Place the financial news CSV in data/raw/ before running this notebook."
    )

## 2. Descriptive Statistics: Headline Length & Publication Trends

In [ ]:
# 2.1 Headline length statistics
print("Headline Length Statistics:")
print(df_news['headline_length'].describe())

# Visualize headline length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_news['headline_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Headline Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Headline Lengths')
axes[0].grid(alpha=0.3)

axes[1].boxplot(df_news['headline_length'])
axes[1].set_ylabel('Headline Length (characters)')
axes[1].set_title('Headline Length Boxplot')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 2.2 Articles per publisher
print("\n\nArticles per Publisher (Top 20):")
publisher_counts = df_news['publisher'].value_counts().head(20)
print(publisher_counts)

fig, ax = plt.subplots(figsize=(12, 6))
publisher_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Number of Articles')
ax.set_ylabel('Publisher')
ax.set_title('Top 20 Publishers by Article Count')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# 2.3 Publication date trends
print("\n\nPublication Date Range:")
print(f"From: {df_news['date'].min()}")
print(f"To: {df_news['date'].max()}")

# Articles per day
articles_per_day = publication_counts_by_day(df_news)

fig, ax = plt.subplots(figsize=(14, 5))
articles_per_day.plot(ax=ax, color='darkblue', linewidth=1.5)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Articles')
ax.set_title('Daily Publication Frequency Over Time')
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Text Analysis: Keywords and Topic Extraction

In [ ]:
# Clean headlines and add derived features
# This also creates the headline_length column used throughout the notebook
df_news = add_headline_features(df_news)

# 3.1 Extract top keywords using TF-IDF
print("=== Top Keywords by TF-IDF ===\n")

top_keywords_tfidf = top_keywords(df_news['headline_clean'], max_features=20, ngram_range=(1, 1))

print("Top 20 Keywords (TF-IDF):")
for keyword, score in top_keywords_tfidf.items():
    print(f"  {keyword:20} {score:8.4f}")

# Visualize top keywords
fig, ax = plt.subplots(figsize=(12, 6))
top_keywords_tfidf.sort_values().plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('TF-IDF Score')
ax.set_title('Top 20 Keywords by TF-IDF Score')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

# 3.2 Bigrams and common phrases
print("\n=== Common Bigrams (2-word phrases) ===\n")

top_bigrams = common_phrases(df_news['headline_clean'], max_features=15, ngram_range=(2, 2))

print("Top 15 Bigrams:")
for bigram, score in top_bigrams.items():
    print(f"  {bigram:30} {score:8.4f}")

# Visualize top bigrams
fig, ax = plt.subplots(figsize=(12, 6))
top_bigrams.sort_values().plot(kind='barh', ax=ax, color='skyblue')
ax.set_xlabel('Count')
ax.set_title('Top 15 Common Phrases (Bigrams)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 4. Publisher Analysis

In [ ]:
# 4.1 Publisher activity summary
print("=== Publisher Statistics ===\n")
print(f"Total unique publishers: {df_news['publisher'].nunique()}")
print(f"\nTop 20 publishers by article count:")
print(df_news['publisher'].value_counts().head(20))

# 4.2 Extract domains if publisher contains email addresses
print("\n=== Domain Analysis (if email addresses are present) ===\n")

df_news['domain'] = df_news['publisher'].map(extract_domain)

if df_news['domain'].notna().sum() > 0:
    print(f"Publishers with email addresses: {df_news['domain'].notna().sum()}")
    print("\nTop domains by article count:")
    domain_counts = df_news['domain'].value_counts().head(15)
    print(domain_counts)
    
    # Visualize domain distribution
    fig, ax = plt.subplots(figsize=(12, 6))
    domain_counts.plot(kind='barh', ax=ax, color='lightgreen')
    ax.set_xlabel('Number of Articles')
    ax.set_ylabel('Domain')
    ax.set_title('Top 15 Domains by Article Count')
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print("No email-style publisher names detected.")

# 4.3 Articles per stock symbol
print("\n=== Stock Coverage ===\n")
print(f"Total unique stocks: {df_news['stock'].nunique()}")
print(f"\nTop 20 stocks by article count:")
stock_counts = df_news['stock'].value_counts().head(20)
print(stock_counts)

fig, ax = plt.subplots(figsize=(12, 6))
stock_counts.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_xlabel('Number of Articles')
ax.set_ylabel('Stock Symbol')
ax.set_title('Top 20 Stocks by Article Count')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 5. Time-of-Day Analysis (if timestamps are available)

In [ ]:
# Extract hour from timestamp
if 'date' in df_news.columns:
    df_news['hour'] = pd.to_datetime(df_news['date'], errors='coerce').dt.hour
    
    # Articles by hour of day
    articles_by_hour = publication_counts_by_hour(df_news)
    
    fig, ax = plt.subplots(figsize=(14, 5))
    articles_by_hour.plot(kind='bar', ax=ax, color='teal')
    ax.set_xlabel('Hour of Day (UTC-4)')
    ax.set_ylabel('Number of Articles')
    ax.set_title('Publication Distribution by Hour of Day')
    ax.set_xticklabels([f'{h}:00' for h in articles_by_hour.index], rotation=45)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print(f"Peak publishing hour: {articles_by_hour.idxmax()}:00")
    print(f"Slowest hour: {articles_by_hour.idxmin()}:00")

## 6. Summary & Key Findings

### EDA Summary Template

**Document your key findings here:**

1. **Dataset Overview**: [Total articles, date range, unique stocks, publishers]
2. **Headline Characteristics**: [Average length, length distribution, notable patterns]
3. **Top Keywords**: [List recurring themes—earnings, FDA, price target, etc.]
4. **Publisher Analysis**: [Most active sources, domain breakdown if applicable]
5. **Temporal Patterns**: [Peak publishing hours, volume spikes around events]
6. **Stock Coverage**: [Which stocks dominate the news, relative coverage]

**Observations & Insights**:
- [Insert your analysis here as you run the notebook]

**Next Steps for Task 2**:
- Merge this branch into main via PR
- Begin technical indicator analysis on historical stock prices
